# Retail dbt Test Generation Pipeline

**Purpose:** Automated end-to-end pipeline that:
1. Extracts schema metadata from OpenMetadata (after ingesting a PostgreSQL retail database)
2. Computes column-level data quality statistics (null rates, distinct counts, zero counts)
3. Sends schema + sample data to a local LLM (Ollama `qwen2.5-coder:3b`) to auto-generate dbt tests
4. Runs the generated dbt tests against the retail database and previews results

---

## Prerequisites

```bash
# 1. Start docker services (PostgreSQL 17, OpenMetadata 1.5.14, Elasticsearch)
cd .. && docker compose up -d

# 2. Install Python dependencies
pip install -r requirements.txt

# 3. Seed the retail database (~143K rows across 14 tables)
python db/seed.py
```

---

## Pipeline Overview

| Cell | What it does |
|------|-------------|
| 1 | Setup, imports, connectivity checks |
| 2 | Register PostgreSQL in OpenMetadata + trigger ingestion |
| 3 | Extract table schemas from OpenMetadata API |
| 4 | Sample data preview (5 rows per table) |
| 5 | Column statistics: null rate, uniqueness, zeros, min/max |
| 6 | LLM test generation via Ollama (`qwen2.5-coder:3b`) |
| 7 | Write generated dbt YAML + SQL models to project |
| 8 | Run dbt: deps → run → test → docs generate |
| 9 | Results dashboard: pass/fail per test |
| 10 | Charts: test status bar + top null-rate columns |
| 11 | Summary + start dbt docs server (port 8082) |

---
## 1 — Setup & Connectivity

### What this cell does
- Adds the project root to `sys.path` so `pipeline.*` modules are importable
- Loads `.env` from the project root (overrides any shell env vars)
- Applies `nest_asyncio` — required because Jupyter already runs an event loop,
  and `asyncio.run()` raises `RuntimeError` without this patch
- Performs live connectivity checks for PostgreSQL, OpenMetadata, and Ollama

> **Expected:** `[OK]` for all three services before proceeding to Cell 2.

In [ ]:
import sys
import os
import asyncio
import json
import subprocess

# nest_asyncio patches the running event loop so asyncio.run() works inside Jupyter.
# Without this, asyncio.run() raises: RuntimeError: This event loop is already running.
import nest_asyncio
nest_asyncio.apply()

import psycopg2
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# ── Add project root to sys.path so `from pipeline.X import Y` works ─────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load .env BEFORE reading os.getenv — this overrides any inherited shell env vars
# (important: the datalens parent project may have conflicting OM_HOST etc.)
from dotenv import load_dotenv
env_path = os.path.join(PROJECT_ROOT, '.env')
load_dotenv(env_path, override=True)   # override=True ensures .env wins over shell
print(f'Loaded env from: {env_path}')

# ── PostgreSQL connection parameters ─────────────────────────────────────────
PG_HOST     = os.getenv('POSTGRES_HOST',     'localhost')
PG_PORT     = int(os.getenv('POSTGRES_PORT', '5435'))
PG_USER     = os.getenv('POSTGRES_USER',     'retail')
PG_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'retail123')
PG_DB       = os.getenv('POSTGRES_DB',       'retail')

# ── OpenMetadata connection parameters ───────────────────────────────────────
# OM_HOST must be 'localhost' and OM_PORT the mapped port (8588), not the
# internal container port (8585). The .env file sets these correctly.
OM_HOST     = os.getenv('OM_HOST',          'localhost')
OM_PORT     = int(os.getenv('OM_PORT',      '8588'))
OM_USER     = os.getenv('OM_ADMIN_USER',    'admin')
OM_PASSWORD = os.getenv('OM_ADMIN_PASSWORD','admin')

# ── Ollama (local LLM) parameters ────────────────────────────────────────────
OLLAMA_HOST  = os.getenv('OLLAMA_HOST',  'localhost')
OLLAMA_PORT  = int(os.getenv('OLLAMA_PORT', '11434'))
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'qwen2.5-coder:3b')

# Absolute path to the dbt project; used by all dbt subprocess calls
DBT_PROJECT_DIR = os.path.join(PROJECT_ROOT, 'dbt_project')

print(f'Config loaded:')
print(f'  PostgreSQL:   {PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DB}')
print(f'  OpenMetadata: http://{OM_HOST}:{OM_PORT}')
print(f'  Ollama:       http://{OLLAMA_HOST}:{OLLAMA_PORT} model={OLLAMA_MODEL}')
print(f'  dbt project:  {DBT_PROJECT_DIR}')
print()

# ── Live connectivity checks ──────────────────────────────────────────────────
checks = {}

# PostgreSQL: attempt a connect + immediate close
try:
    _c = psycopg2.connect(
        host=PG_HOST, port=PG_PORT,
        user=PG_USER, password=PG_PASSWORD,
        dbname=PG_DB, connect_timeout=5
    )
    _c.close()
    checks['PostgreSQL'] = True
except Exception as e:
    checks['PostgreSQL'] = str(e)

# OpenMetadata: hit the public system status endpoint (no auth required)
try:
    r = requests.get(f'http://{OM_HOST}:{OM_PORT}/api/v1/system/status', timeout=5)
    checks['OpenMetadata'] = r.status_code == 200
except Exception as e:
    checks['OpenMetadata'] = str(e)

# Ollama: list available models — returns JSON with 'models' key if running
try:
    r = requests.get(f'http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/tags', timeout=5)
    if r.status_code == 200:
        models = [m['name'] for m in r.json().get('models', [])]
        checks['Ollama'] = f'OK ({len(models)} models: {models})'
    else:
        checks['Ollama'] = f'HTTP {r.status_code}'
except Exception as e:
    checks['Ollama'] = str(e)

print('=== Connectivity Checks ===')
all_ok = True
for service, status in checks.items():
    ok = status is True or (isinstance(status, str) and status.startswith('OK'))
    icon = 'OK  ' if ok else 'FAIL'
    if not ok:
        all_ok = False
    print(f'  [{icon}] {service}: {"connected" if status is True else status}')

print()
if all_ok:
    print('All services reachable. Proceed to Cell 2.')
else:
    print('Fix failing services before continuing.')
    print('  PostgreSQL / OM: run `docker compose up -d` from the project root.')
    print('  Ollama: run `ollama serve` if not already running.')


---
## 2 — OpenMetadata: Service Registration & Metadata Ingestion

### What this cell does
1. Authenticates with OpenMetadata using JWT (via `POST /api/v1/users/login`)
2. Registers the local PostgreSQL `retail` database as a service named **`retail-postgres`**
   (idempotent — skips creation if it already exists)
3. Creates a metadata ingestion pipeline if one doesn't exist
4. Triggers the pipeline and polls every 5 s until it completes (3-minute timeout)
5. Fetches and returns the discovered table list with column metadata

### After this cell
All 14 retail tables will be visible in the OpenMetadata catalogue at
`http://localhost:8588`.

> **Note:** Ingestion typically takes 15–60 seconds depending on host resources.

In [ ]:
from pipeline.om_client import OpenMetadataClient

OM_SERVICE_NAME = 'retail-postgres'   # logical name for the PG service inside OM

async def run_ingestion():
    """
    Full OM flow:
      ensure service exists → ensure pipeline exists → trigger → poll → return tables
    """
    async with OpenMetadataClient(OM_HOST, OM_PORT, OM_USER, OM_PASSWORD) as om:

        # Fail fast if OM server is not reachable
        if not await om.health():
            print('ERROR: OpenMetadata is not healthy.')
            print('  Run `docker compose up -d` from the project root.')
            return None

        print(f'OpenMetadata is healthy. Service: {OM_SERVICE_NAME}')

        # run_full_ingestion handles the complete flow in one call:
        #   get_or_create_service → get_or_create_pipeline → trigger → poll → list_tables
        tables = await om.run_full_ingestion(
            service_name=OM_SERVICE_NAME,
            pg_host=PG_HOST,
            pg_port=PG_PORT,
            pg_db=PG_DB,
            pg_user=PG_USER,
            pg_password=PG_PASSWORD,
        )
        return tables

# asyncio.run() works here because nest_asyncio was applied in Cell 1
om_tables = asyncio.run(run_ingestion())

print()
print(f'Ingestion complete. Tables discovered: {len(om_tables) if om_tables else 0}')


---
## 3 — Schema Extraction

Reads the table list returned by Cell 2 and builds the lookup structures
consumed by all subsequent cells:

| Variable | Type | Description |
|----------|------|-------------|
| `table_names` | `list[str]` | Ordered list of table names |
| `table_columns` | `dict` | `table_name → [column_defs]` — each column has `name`, `dataType`, `nullable` |

In [ ]:
if om_tables:
    # Display a summary table: one row per discovered table
    schema_df = pd.DataFrame([
        {
            'table_name':   t['name'],
            'column_count': len(t['columns']),
            'fqn':          t['fqn'],    # fully-qualified name in OM: service.db.schema.table
        }
        for t in om_tables
    ])
    print('Tables extracted from OpenMetadata:')
    display(schema_df)

    # Build lookup structures used by Cells 4-7
    table_columns = {t['name']: t['columns'] for t in om_tables}
    table_names   = [t['name'] for t in om_tables]
    print(f'\n{len(table_names)} tables ready for profiling and test generation.')

else:
    print('No tables found. Ensure:')
    print('  1. seed.py has been run: python db/seed.py')
    print('  2. OM ingestion completed successfully in Cell 2')
    table_columns = {}
    table_names   = []


---
## 4 — Sample Data Preview

Queries 5 rows from each table so you can visually inspect the raw data
before statistics are computed. Useful for:
- Confirming the seed ran correctly
- Spotting obvious data anomalies before the LLM sees them

> Only the **first 5 tables** are shown to keep output manageable.
> Change `[:5]` in the slice to show more tables.

A persistent database connection (`conn`) is opened here and reused in Cells 5 and 6.

In [ ]:
from pipeline.dbt_generator import get_sample_rows

# Open a persistent connection reused across Cells 4, 5, 6.
# Closed in Cell 11 after the pipeline completes.
conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT,
    user=PG_USER, password=PG_PASSWORD,
    dbname=PG_DB
)

print('Sample data — 5 rows per table (showing first 5 tables)\n')

# Show only the first 5 tables; extend the slice to preview more
for table in table_names[:5]:
    rows = get_sample_rows(conn, table, limit=5)
    if rows:
        print(f'── {table} ({len(rows)} rows shown) ──')
        display(pd.DataFrame(rows))
        print()


---
## 5 — Column Statistics

### Metrics computed per column

| Metric | Formula | Use in LLM prompt |
|--------|---------|-------------------|
| `null_rate` | `null_count / total_rows` | Skip `not_null` test if > 0 |
| `uniqueness_score` | `distinct / non_null` | Add `unique` test if > 0.99 |
| `distinct_count` | `COUNT(DISTINCT col)` | Add `accepted_values` if < 15 |
| `zero_count` | `COUNT(*) WHERE col = 0` | Flag numeric columns with zeros |
| `min_val` / `max_val` | `MIN(col)` / `MAX(col)` | Range checks for numeric cols |

These statistics are embedded in the prompt sent to Ollama in Cell 6,
giving the LLM enough context to make accurate test decisions.

In [ ]:
from pipeline.dbt_generator import compute_column_stats

print('Computing column statistics for all tables...\n')

# all_stats: { table_name: { col_name: { null_rate, uniqueness_score, ... } } }
all_stats = {}

for table in table_names:
    cols = table_columns.get(table, [])
    if not cols:
        continue
    stats = compute_column_stats(conn, table, cols)
    all_stats[table] = stats
    print(f'  {table}: {len(cols)} columns analysed')

# ── Flatten into a single DataFrame for display ───────────────────────────────
stat_rows = []
for table, col_stats in all_stats.items():
    for col_name, s in col_stats.items():
        stat_rows.append({
            'table':            table,
            'column':           col_name,
            'total_rows':       s.get('total_count', 0),
            'null_rate':        round(s.get('null_rate', 0.0), 4),
            'uniqueness_score': round(s.get('uniqueness_score', 0.0), 4),
            'distinct_count':   s.get('distinct_count', 0),
            'zero_count':       s.get('zero_count'),       # None for non-numeric columns
            'min_val':          s.get('min_val'),
            'max_val':          s.get('max_val'),
        })

# Sort by null_rate descending — worst columns at top
stats_df = pd.DataFrame(stat_rows).sort_values('null_rate', ascending=False).reset_index(drop=True)

print(f'\nColumn statistics — top 30 by null rate (expected: email ~4%, cost_price ~4%):')
display(stats_df.head(30))


---
## 6 — LLM Test Generation (Ollama)

### How the prompt is constructed
For each table, `build_prompt()` creates a structured prompt containing:
- Column schema from OpenMetadata (name, type, nullable)
- Per-column statistics from Cell 5 (null rate, uniqueness, distinct count, zeros, min/max)
- Up to 10 sample rows from PostgreSQL

### Test types generated by the LLM

| dbt Test | Trigger condition in prompt |
|----------|-----------------------------|
| `not_null` | null_rate = 0 and nullable = False |
| `unique` | uniqueness_score > 0.99 and column ends in `_id` |
| `accepted_values` | distinct_count < 15 and non-numeric type |
| `dbt_utils.expression_is_true` | numeric amount/price/qty/salary ≥ 0 |
| `dbt_utils.expression_is_true` | rating column: `rating >= 1 and rating <= 5` |

> **Timing:** `qwen2.5-coder:3b` takes ~10–30 s per table on CPU.
> With 14 tables, total generation time is approximately 3–7 minutes.

In [ ]:
from pipeline.llm_client import OllamaClient, build_prompt, parse_yaml_output

# ── Initialise Ollama client ──────────────────────────────────────────────────
ollama = OllamaClient(OLLAMA_HOST, OLLAMA_PORT, OLLAMA_MODEL)

if not ollama.health():
    print(f'ERROR: Ollama not reachable at http://{OLLAMA_HOST}:{OLLAMA_PORT}')
    print('Run: ollama serve')
else:
    print(f'Ollama ready — model: {OLLAMA_MODEL}\n')

# Accumulates one parsed YAML dict per table; merged in Cell 7
per_table_schemas = []

for table in table_names:
    cols        = table_columns.get(table, [])
    stats       = all_stats.get(table, {})

    # Fetch up to 10 sample rows to include actual data in the prompt
    sample_rows = get_sample_rows(conn, table, limit=10)

    # Build structured prompt: column schema + stats table + sample rows
    prompt = build_prompt(table, cols, stats, sample_rows)

    print(f'Generating tests for [{table}]...')
    raw = ollama.generate(prompt)     # synchronous REST call to Ollama

    # parse_yaml_output strips markdown fences and parses the YAML string
    parsed = parse_yaml_output(raw)

    if parsed:
        models = parsed.get('models', [])
        # Count individual test entries across all columns of all models
        test_count = sum(
            len(c.get('tests', []))
            for m in models
            for c in m.get('columns', [])
        )
        per_table_schemas.append(parsed)
        print(f'  OK — {len(models)} model(s), {test_count} tests generated')
    else:
        # Log the first 200 chars so the failure reason is visible
        print(f'  WARN: could not parse YAML for [{table}]')
        print(f'  Raw snippet: {raw[:200]}')

print(f'\nDone. Schemas generated for {len(per_table_schemas)}/{len(table_names)} tables.')


---
## 7 — Write dbt Files

Merges all per-table schema dicts into a single `schema.yml` and writes
the following files into `dbt_project/models/staging/`:

| File | Content |
|------|--------|
| `sources.yml` | dbt source definitions — points to PostgreSQL `public` schema |
| `schema.yml` | **LLM-generated** tests for all staging models |
| `stg_{table}.sql` | Thin staging view: `SELECT * FROM {{ source('retail', '{table}') }}` |

> Re-running this cell overwrites previously generated files — safe to repeat.

In [ ]:
from pipeline.dbt_generator import write_dbt_files, merge_schema_ymls

# Merge all per-table YAML dicts into one combined schema.yml string
merged_yml = merge_schema_ymls(per_table_schemas)

print(f'Writing dbt files to: {DBT_PROJECT_DIR}/models/staging/')
print()

write_dbt_files(
    dbt_project_dir=DBT_PROJECT_DIR,
    tables=table_names,
    merged_schema_yml=merged_yml,
    source_name='retail',     # must match the source name in sources.yml
)

print()
print('Done. Tip: open dbt_project/models/staging/schema.yml to review generated tests.')


---
## 8 — Run dbt

### Execution order

| Step | Command | Purpose |
|------|---------|--------|
| 1 | `dbt deps` | Install `dbt_utils` and `dbt_expectations` from `packages.yml` |
| 2 | `dbt run --select staging` | Materialise all `stg_*` views in the `retail_staging` schema |
| 3 | `dbt test --store-failures --select staging` | Run generated tests; store failures in `retail.dbt_test__audit.*` |
| 4 | `dbt docs generate` | Build `catalog.json` + `manifest.json` for the docs site |

### Expected outcome
Some tests will intentionally **fail** — the seed data contains known DQ issues:
- Duplicate `order_code` values → `unique` test fails on `orders`
- NULL `email` on 200 customers → `not_null` test fails on `customers`
- `line_total = 0` on 300 order items → expression test fails on `order_items`
- Out-of-range `rating` on 20 reviews → range test fails on `product_reviews`

These failures are by design — they make the test generation result meaningful.

In [ ]:
def run_dbt(args: list, cwd: str) -> int:
    """Run a dbt command, stream truncated output, return exit code."""
    cmd = ['dbt'] + args + ['--profiles-dir', '.']
    print(f'\n>>> dbt {" ".join(args)}')
    print('-' * 60)
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)

    # Truncate stdout to last 3000 chars — dbt output can be verbose
    stdout = result.stdout.strip()
    if len(stdout) > 3000:
        print('[ ... output truncated ... ]\n')
    print(stdout[-3000:])

    # Only show stderr on failure
    if result.returncode != 0 and result.stderr:
        print('STDERR:', result.stderr[-500:])

    print(f'Exit code: {result.returncode}')
    return result.returncode


# Step 1 — install dbt packages (dbt_utils, dbt_expectations)
run_dbt(['deps'], DBT_PROJECT_DIR)

# Step 2 — materialise staging views in PostgreSQL
run_dbt(['run', '--select', 'staging'], DBT_PROJECT_DIR)

# Step 3 — run generated tests; failures written to retail.dbt_test__audit schema
run_dbt(['test', '--store-failures', '--select', 'staging'], DBT_PROJECT_DIR)

# Step 4 — generate docs artifacts (catalog.json, manifest.json, run_results.json)
run_dbt(['docs', 'generate'], DBT_PROJECT_DIR)

print('\n=== dbt pipeline complete. Proceed to Cell 9 for results. ===')


---
## 9 — Results Dashboard

Parses `target/run_results.json` produced by `dbt test` and displays a
sortable table of every test with its status, execution time, and failure count.

### Status legend

| Status | Meaning |
|--------|--------|
| `pass` | Zero rows violated the constraint |
| `fail` | At least one row violated the constraint |
| `warn` | Soft threshold exceeded (if configured) |
| `error` | dbt could not compile or execute the test SQL |

In [ ]:
run_results_path = os.path.join(DBT_PROJECT_DIR, 'target', 'run_results.json')

if not os.path.exists(run_results_path):
    print('run_results.json not found — run Cell 8 first.')
else:
    with open(run_results_path) as f:
        run_results = json.load(f)

    result_rows = []
    for r in run_results.get('results', []):
        # unique_id format: test.retail_poc.<test_name>.<hash>
        uid       = r.get('unique_id', '')
        node_name = uid.split('.')[-1]   # last segment is the human-readable test name

        result_rows.append({
            'test_name':        node_name,
            'status':           r.get('status', 'unknown'),
            'failures':         r.get('failures', 0),
            'execution_time_s': round(r.get('execution_time', 0), 2),
            'message':          (r.get('message') or '')[:80],
        })

    # Sort: failed first, then by failure count descending — worst offenders at top
    results_df = (
        pd.DataFrame(result_rows)
          .sort_values(['status', 'failures'], ascending=[True, False])
          .reset_index(drop=True)
    )

    # Summary counts
    total  = len(results_df)
    passed = (results_df['status'] == 'pass').sum()
    failed = (results_df['status'] == 'fail').sum()
    errors = (results_df['status'] == 'error').sum()

    print('=== dbt Test Results ===')
    print(f'  Total:   {total}')
    print(f'  Passed:  {passed}')
    print(f'  Failed:  {failed}  ← expected (seeded DQ issues)')
    if errors:
        print(f'  Errors:  {errors}  ← investigate: likely YAML syntax issues in schema.yml')
    print()
    display(results_df)


---
## 10 — Visualisation

Two side-by-side charts:

1. **dbt Tests by Status** — bar chart (green = pass, red = fail, orange = warn, purple = error)
2. **Top 10 Columns by Null Rate** — horizontal bar; pinpoints the worst data quality issues

The chart is saved to `retail-dbt-poc/dbt_test_results.png`.

In [ ]:
if 'results_df' not in dir() or results_df.empty:
    print('No results to chart. Run Cell 9 first.')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # ── Chart 1: dbt test results by status ──────────────────────────────────
    status_counts = results_df['status'].value_counts()
    status_colors = {
        'pass':  '#4CAF50',   # green
        'fail':  '#F44336',   # red
        'warn':  '#FF9800',   # orange
        'error': '#9C27B0',   # purple
    }
    bar_colors = [status_colors.get(s, '#9E9E9E') for s in status_counts.index]

    ax1.bar(status_counts.index, status_counts.values, color=bar_colors, edgecolor='white', linewidth=0.8)
    ax1.set_title('dbt Tests by Status', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Status')
    ax1.set_ylabel('Number of Tests')

    # Annotate each bar with its count
    for i, (_, val) in enumerate(status_counts.items()):
        ax1.text(i, val + 0.15, str(val), ha='center', va='bottom', fontweight='bold', fontsize=11)

    # ── Chart 2: top 10 columns by null rate ─────────────────────────────────
    if 'stats_df' in dir() and not stats_df.empty:
        # Filter to columns that actually have nulls, then take top 10
        top_nulls = stats_df[stats_df['null_rate'] > 0].nlargest(10, 'null_rate')

        if not top_nulls.empty:
            labels = top_nulls['table'] + '.' + top_nulls['column']
            ax2.barh(
                labels,
                top_nulls['null_rate'] * 100,
                color='#F44336', alpha=0.75, edgecolor='white'
            )
            ax2.set_title('Top 10 Columns by Null Rate', fontsize=13, fontweight='bold')
            ax2.set_xlabel('Null Rate (%)')
            ax2.invert_yaxis()   # highest null rate at the top
        else:
            ax2.text(0.5, 0.5, 'No nulls detected!',
                     ha='center', va='center', transform=ax2.transAxes, fontsize=12)
            ax2.set_title('Null Rates', fontsize=13)

    plt.suptitle('Retail dbt POC — Data Quality Overview', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()

    # Save chart to project root
    chart_path = os.path.join(PROJECT_ROOT, 'dbt_test_results.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Chart saved → {chart_path}')


---
## 11 — Summary & dbt Docs

Starts `dbt docs serve` on port **8082** as a background process so you can
browse the auto-generated documentation while reviewing results in the notebook.

The docs site shows:
- All staging models with column descriptions
- LLM-generated tests per column
- Test pass/fail status from the last run
- Source → model lineage

### Service URLs

| Service | URL |
|---------|-----|
| **dbt Docs** | http://localhost:8082 |
| OpenMetadata | http://localhost:8588 |
| Elasticsearch | http://localhost:9202 |
| Ollama API | http://localhost:11434 |
| PostgreSQL | `localhost:5435` (user: retail, db: retail) |

In [ ]:
# Start dbt docs serve as a non-blocking background process.
# The process is kept in `docs_proc` so you can terminate it later.
docs_proc = subprocess.Popen(
    ['dbt', 'docs', 'serve', '--port', '8082', '--profiles-dir', '.'],
    cwd=DBT_PROJECT_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# ── Final summary ─────────────────────────────────────────────────────────────
print('=' * 55)
print('   RETAIL DBT TEST GENERATION PIPELINE — COMPLETE')
print('=' * 55)
print(f'  Tables ingested from OpenMetadata : {len(table_names)}')
print(f'  dbt schemas generated by Ollama   : {len(per_table_schemas)}')

if 'results_df' in dir() and not results_df.empty:
    passed = (results_df['status'] == 'pass').sum()
    failed = (results_df['status'] == 'fail').sum()
    total  = len(results_df)
    print(f'  dbt tests run                     : {total}')
    print(f'    Passed                           : {passed}')
    print(f'    Failed (expected DQ issues)      : {failed}')

print()
print('  dbt Docs site  →  http://localhost:8082')
print('  OpenMetadata   →  http://localhost:8588')
print()
print('  To stop the dbt docs server:')
print('    docs_proc.terminate()')

# Close the persistent DB connection opened in Cell 4
if 'conn' in dir() and conn and not conn.closed:
    conn.close()
    print()
    print('  Database connection closed.')
